In [8]:
import pandas as pd
import numpy as np



In [21]:
X_train = pd.read_csv('X_train_resampled.csv')
X_test = pd.read_csv('X_test.csv').drop(columns=["x", "y","z"])
y_train = pd.read_csv('y_train_resampled.csv')
y_test = pd.read_csv('y_test.csv')

In [22]:
X_train.drop(columns=["x", "y","z"], inplace=True)

In [23]:
X_train

,stresses_full_xx,stresses_full_xy,stresses_full_xz,stresses_full_yy,stresses_full_yz,stresses_full_zz,grid_aftershock_count,Magnitudes_avg
0,6.974558e+03,6.124351e+03,-4.746838e+03,2.008698e+03,-3.685083e+03,2111.907848,0,0.000000
1,1.503606e+03,-3.075157e+03,2.933184e+02,-1.774207e+03,-2.619814e+03,-1718.674774,0,0.000000
2,-7.553082e+04,5.589479e+04,2.617920e+04,-1.206959e+05,-3.465612e+04,-7375.982482,0,0.000000
3,-1.140306e+03,-2.784073e+03,1.128363e+03,7.178003e+03,-5.024222e+03,2216.158344,0,0.000000
4,-3.819540e+04,1.285215e+04,-3.934917e+04,-4.096452e+04,-5.789952e+04,-53216.088967,0,0.000000
...,...,...,...,...,...,...,...,...
7074005,1.018179e+05,4.139885e+05,-2.704068e+05,-8.250044e+05,-3.019438e+05,310207.543554,1,2.782521
7074006,3.563715e+05,2.371107e+05,8.181173e+04,3.705232e+05,1.328532e+05,11595.058872,1,3.470446
7074007,1.531967e+06,7.470506e+05,9.197640e+05,1.056978e+06,7.776117e+05,718000.263455,1,4.399577
7074008,5.431770e+06,-4.380702e+06,2.326016e+06,5.273890e+06,-1.202699e+06,251677.300131,1,3.828475


In [24]:
import tensorflow as tf

In [25]:
X_test

,stresses_full_xx,stresses_full_xy,stresses_full_xz,stresses_full_yy,stresses_full_yz,stresses_full_zz,grid_aftershock_count,Magnitudes_avg
0,6.744308e+05,-166529.185099,4.830643e+05,473527.100136,913546.814229,1.000908e+06,0,0.0
1,1.362078e+04,-6659.833747,-5.473842e+03,-15379.726480,1578.292566,-2.213988e+03,0,0.0
2,-1.234689e+05,-49113.079755,-4.667018e+04,-44850.134054,-115813.998222,-2.050637e+04,0,0.0
3,-3.602603e+04,-26613.853905,-1.030677e+04,-33098.411012,-10141.886594,-8.425329e+02,0,0.0
4,9.854980e+03,7172.217336,-4.129090e+03,2211.552280,-2080.774843,1.010909e+03,0,0.0
...,...,...,...,...,...,...,...,...
902050,-5.893799e+03,-4974.114458,-6.657127e+03,-10892.272089,695.092088,-3.716064e+03,0,0.0
902051,1.412257e+06,44739.430284,1.864080e+06,344559.637500,227836.545828,-1.281043e+06,0,0.0
902052,-1.196460e+04,-5733.610289,3.204576e+03,-7091.149108,7558.726356,-2.587940e+03,0,0.0
902053,3.273277e+04,-20536.415468,-1.680455e+04,37594.374291,22105.187045,1.031546e+04,0,0.0


In [26]:
#scale data
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train

array([[0.29333839, 0.29203318, 0.41390949, ..., 0.74569011, 0.        ,
        0.        ],
       [0.29333787, 0.29202983, 0.41390966, ..., 0.74568974, 0.        ,
        0.        ],
       [0.29333059, 0.29205131, 0.41391054, ..., 0.7456892 , 0.        ,
        0.        ],
       ...,
       [0.2934826 , 0.29230311, 0.41394087, ..., 0.74575834, 0.01111111,
        0.48400185],
       [0.29385137, 0.29043502, 0.4139886 , ..., 0.74571389, 0.01111111,
        0.42117441],
       [0.29331559, 0.29209551, 0.41392422, ..., 0.74572021, 0.01111111,
        0.46222537]])

In [33]:
from tqdm.autonotebook import tqdm

C:\Users\ismailbgr\AppData\Local\Temp\ipykernel_28148\987820437.py:1: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [45]:
import torch
from torch.utils.data import DataLoader, TensorDataset

import torch.nn as nn
import torch.optim as optim

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define the neural network
class ANN(nn.Module):
    def __init__(self, input_size):
        super(ANN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

# Split for validation (20%)
dataset_size = len(X_train_tensor)
indices = torch.randperm(dataset_size)
train_size = int(dataset_size * 0.8)
train_indices = indices[:train_size]
val_indices = indices[train_size:]

# Create data loaders
train_dataset = TensorDataset(X_train_tensor[train_indices], y_train_tensor[train_indices])
val_dataset = TensorDataset(X_train_tensor[val_indices], y_train_tensor[val_indices])
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512)

# Initialize model
model = ANN(X_train.shape[1]).to(device) #0.0000005
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(),lr=0.0000001)

# Training loop
num_epochs = 4
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in tqdm(range(num_epochs), desc="Training Epochs"):
    # Training
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    for inputs, targets in tqdm(train_loader, desc="Training Batches"):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        predicted = (outputs > 0.5).float()
        train_total += targets.size(0)
        train_correct += (predicted == targets).sum().item()
    
    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            predicted = (outputs > 0.5).float()
            val_total += targets.size(0)
            val_correct += (predicted == targets).sum().item()
    
    # Calculate metrics
    train_loss /= train_total
    train_acc = train_correct / train_total
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

Using device: cuda


Training Epochs:   0%|          | 0/4 [00:00<?, ?it/s]

Training Batches:   0%|          | 0/11054 [00:00<?, ?it/s]

Epoch 1/4, Train Loss: 0.6914, Train Acc: 0.5001, Val Loss: 0.6889, Val Acc: 0.4996


Training Batches:   0%|          | 0/11054 [00:00<?, ?it/s]

Epoch 2/4, Train Loss: 0.6841, Train Acc: 0.7063, Val Loss: 0.6778, Val Acc: 0.9402


Training Batches:   0%|          | 0/11054 [00:00<?, ?it/s]

Epoch 3/4, Train Loss: 0.6681, Train Acc: 0.9695, Val Loss: 0.6570, Val Acc: 0.9825


Training Batches:   0%|          | 0/11054 [00:00<?, ?it/s]

Epoch 4/4, Train Loss: 0.6423, Train Acc: 0.9845, Val Loss: 0.6255, Val Acc: 0.9851


In [46]:
#predict the test set and print the f1 score
model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_pred = model(X_test_tensor)
    y_pred = (y_pred > 0.5).float().cpu().numpy()
    y_pred = pd.DataFrame(y_pred, columns=["prediction"])


In [48]:
#calculate f1 score
import numpy as np
from sklearn.metrics import f1_score



In [49]:
f1_micro = f1_score(y_test, y_pred, average='micro')
print(f"F1 Score (Micro Average): {f1_micro:.4f}")

F1 Score (Micro Average): 0.9993
